In [1]:
import numpy as np
import pandas as pd
import pydicom
import os
import scipy.ndimage
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from skimage import measure, morphology
from sklearn.preprocessing import normalize

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset




In [2]:
def csv_preprocess(data):

    data["Healthy-FVC"] = round((data["FVC"] * 100) / data["Percent"])
    FE = []
    FE.append("Healthy-FVC")

    COLS = ["Sex", "SmokingStatus"]
    for col in COLS:
        for mod in data[col].unique():
            FE.append(mod)
            data[mod] = (data[col] == mod).astype(int)

    data = data[["Patient", "Weeks", "FVC", "Age"] + FE]
    data = data.sort_values(["Patient", "Weeks"], ascending=True).reset_index(drop=True)

    FE1 = ["Male", "Female", "Ex-smoker", "Never smoked", "Currently smokes"]
    rename_col = {"Weeks": "base_Weeks", "FVC": "base_FVC"}
    data = data.rename(columns=rename_col)

    npData = pd.DataFrame(
        columns=["Patient", "base_Weeks", "base_FVC", "Age"]
        + FE1
        + ["Week", "Healthy-FVC", "actual_FVC"]
    )

    for pid in data["Patient"].unique():
        weeks = data.loc[data["Patient"] == pid].base_Weeks
        fvc = data.loc[data["Patient"] == pid].base_FVC
        index = data.loc[data["Patient"] == pid].index
        weeks.reset_index(inplace=True, drop=True)
        fvc.reset_index(inplace=True, drop=True)
        for k in range(len(weeks)):
            npData = pd.concat(
                [npData, data.loc[data.index == index[0]]],
                sort=False,
                ignore_index=True,
            )
            npData.iloc[-1, npData.columns.get_loc("Week")] = weeks[k]
            npData.iloc[-1, npData.columns.get_loc("actual_FVC")] = fvc[k]
    npData.reset_index(inplace=True, drop=True)
    npData = npData.fillna(0)

    return npData




In [3]:
class SIGMA(nn.Module):
    def __init__(self):
        super(SIGMA, self).__init__()
        self.data_net1 = nn.Sequential(
            nn.Linear(10, 42),
            nn.ReLU(),
            nn.Linear(42, 64),
            nn.ReLU(),
            nn.Linear(64, 118),
            nn.ReLU(),
        )
        self.data_net2 = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 502), nn.ReLU()
        )
        self.data_net3 = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 118), nn.ReLU()
        )
        self.data_net4 = nn.Sequential(
            nn.Linear(748, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
            nn.ReLU(),
        )

    def forward(self, data_i):
        out1 = self.data_net1(data_i)
        out2 = torch.cat((data_i, out1), dim=-1)
        out2 = self.data_net2(out2)
        out3 = torch.cat((data_i, out2), dim=-1)
        out3 = self.data_net3(out3)
        out4 = torch.cat((data_i, out1, out2, out3), dim=-1)
        out = self.data_net4(out4)
        return out




In [4]:
C1, C2 = torch.tensor(70, dtype=torch.float32), torch.tensor(1000, dtype=torch.float32)


def score(y_true, y_pred):
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]

    delta = (y_true[:, 0] - fvc_pred).abs()

    sq2 = torch.tensor(2.0).sqrt()
    metric = (delta / sigma) * sq2 + (sigma * sq2).log()
    return (metric).mean()


def closs(y_true, y_pred):
    sigma = y_pred[:, 2] - y_pred[:, 0]
    e = torch.abs(y_true - y_pred[:, 1])
    loss = torch.abs(sigma - e)
    return loss


def quartile_loss(y_true, y_pred):  # 0.65
    loss = score(y_true, y_pred)  # 0.35
    return loss




In [5]:
def make_eval_data(npEval, model, device="cuda"):
    x_features = npEval[
        [
            "base_Weeks",
            "base_FVC",
            "Age",
            "Male",
            "Female",
            "Ex-smoker",
            "Never smoked",
            "Currently smokes",
            "Week",
            "Healthy-FVC",
        ]
    ]
    x_features = torch.tensor(x_features.values).float()
    x_patientids_name = npEval[["Patient"]].values

    if torch.cuda.is_available() and device == "cuda":
        model.to("cuda")
    else:
        model.to("cpu")
    model.eval()
    predictions = []
    with torch.no_grad():
        for i, patientid in enumerate(x_patientids_name):
            x_feature = x_features[i].unsqueeze(0)
            if torch.cuda.is_available() and device == "cuda":
                x_feature = x_feature.cuda()
            prediction = model(x_feature)
            predictions.append(prediction.to("cpu").detach().numpy()[0])

    predictions = np.array(predictions)
    npEval["FVC"] = predictions[:, 1]
    npEval["Confidence"] = predictions[:, 2] - predictions[:, 0]

    return npEval




In [6]:
data_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
data_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)



In [7]:
submission["Patient"] = submission["Patient_Week"].apply(lambda x: x.split("_")[0])
submission["Weeks"] = (
    submission["Patient_Week"].apply(lambda x: x.split("_")[1]).astype(int)
)
submission = submission.sort_values(
    by=["Patient", "Weeks"], ascending=True
).reset_index(drop=True)



In [8]:
merge = (
    pd.merge(data_test, submission, on=["Patient"], how="left")
    .sort_values(["Patient", "Weeks_y"])
    .reset_index(drop=True)
)
merge = merge.drop(["FVC_y"], axis=1)
merge = merge.rename(
    columns={"FVC_x": "base_FVC", "Weeks_y": "Week", "Weeks_x": "base_Weeks"}
)

del data_test
del submission

data_test = merge.loc[
    :,
    [
        "Patient",
        "base_Weeks",
        "base_FVC",
        "Percent",
        "Age",
        "Sex",
        "SmokingStatus",
        "Week",
    ],
]
submission = merge.loc[:, ["Patient_Week", "base_FVC", "Confidence"]]
submission = submission.rename(columns={"base_FVC": "FVC"})



In [9]:
# Fix pandas>=2.0 removal of DataFrame.append (runtime bug fix; score-neutral).
data = data_test.copy()
data["Healthy-FVC"] = round((data["base_FVC"] * 100) / data["Percent"])
FE = []
FE.append("Healthy-FVC")

COLS = ["Sex", "SmokingStatus"]
for col in COLS:
    for mod in data[col].unique():
        FE.append(mod)
        data[mod] = (data[col] == mod).astype(int)

FE1 = ["Male", "Female", "Ex-smoker", "Never smoked", "Currently smokes"]
npData = pd.DataFrame(
    columns=["Patient", "base_Weeks", "base_FVC", "Age", "Healthy-FVC"] + FE1 + ["Week"]
)

# was: npData = npData.append(data, sort=True)
npData = pd.concat([npData, data], axis=0, ignore_index=True, sort=True)

npData = npData.fillna(0)

del data_test, data
data_test = npData[
    [
        "Patient",
        "base_Weeks",
        "base_FVC",
        "Age",
        "Healthy-FVC",
        "Male",
        "Female",
        "Ex-smoker",
        "Never smoked",
        "Currently smokes",
        "Week",
    ]
]
del npData




/tmp/ipykernel_55/3765152830.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  npData = pd.concat([npData, data], axis=0, ignore_index=True, sort=True)
/tmp/ipykernel_55/3765152830.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  npData = npData.fillna(0)


In [10]:
# Checkpoint path doesn't exist in this environment; fallback to training the same model on tabular features.
def _build_train_dataframe(data_train_raw: pd.DataFrame) -> pd.DataFrame:
    npTrain = csv_preprocess(data_train_raw.copy())
    # Align columns to exactly those used at inference
    xcols = [
        "base_Weeks",
        "base_FVC",
        "Age",
        "Male",
        "Female",
        "Ex-smoker",
        "Never smoked",
        "Currently smokes",
        "Week",
        "Healthy-FVC",
    ]
    ycols = ["actual_FVC"]
    for c in xcols + ycols:
        if c not in npTrain.columns:
            npTrain[c] = 0
    npTrain = npTrain.fillna(0)
    return npTrain, xcols, ycols


def _train_sigma_fallback(
    data_train_raw: pd.DataFrame,
    device: str = "cuda",
    epochs: int = 80,
    batch_size: int = 256,
    lr: float = 1e-3,
) -> SIGMA:
    npTrain, xcols, ycols = _build_train_dataframe(data_train_raw)
    X = torch.tensor(npTrain[xcols].values, dtype=torch.float32)
    y = torch.tensor(npTrain[ycols].values, dtype=torch.float32)

    ds = TensorDataset(X, y)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    model = SIGMA()
    if torch.cuda.is_available() and device == "cuda":
        model = model.cuda()
    else:
        device = "cpu"
        model = model.cpu()

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            if device == "cuda":
                xb = xb.cuda(non_blocking=True)
                yb = yb.cuda(non_blocking=True)

            pred = model(xb)

            # Ensure sigma positive and not tiny to avoid div/0 in loss; preserves core semantics (sigma derived from outputs).
            sigma = (pred[:, 2] - pred[:, 0]).clamp(min=1.0)
            fvc_pred = pred[:, 1]
            delta = (yb[:, 0] - fvc_pred).abs()

            sq2 = torch.tensor(2.0, device=pred.device).sqrt()
            loss = ((delta / sigma) * sq2 + torch.log(sigma * sq2)).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    model.eval()
    return model


ckpt_path = "../input/18-680/Epoch18_Score6.809349381844729_Acc0.9311210927979046.pth"
model = SIGMA()

if os.path.exists(ckpt_path):
    # Bug fix: use map_location so loading works on CPU-only too.
    map_loc = "cuda" if torch.cuda.is_available() else "cpu"
    state = torch.load(ckpt_path, map_location=map_loc)
    model.load_state_dict(state)
else:
    model = _train_sigma_fallback(data_train, device="cuda")

test = make_eval_data(data_test.copy(), model, device="cuda")



/tmp/ipykernel_55/1704154811.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  npData = pd.concat(
/tmp/ipykernel_55/1704154811.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  npData = npData.fillna(0)


In [11]:
# FVC and confidence correction at base week (as original); guard for safety if base week row isn't present.
for nid in test.Patient.unique():
    idx = test[(test.Patient == nid) & (test.Week == test.base_Weeks)].index.values
    if len(idx) > 0:
        test.iloc[idx[0], test.columns.get_loc("FVC")] = test.iloc[
            idx[0], test.columns.get_loc("base_FVC")
        ]
        test.iloc[idx[0], test.columns.get_loc("Confidence")] = 70



In [12]:
# Clip confidence as required by competition metric.
test.loc[test.Confidence < 70, "Confidence"] = 70

# Ensure submission row alignment matches the original sample_submission order (already preserved via merge sorting).
submission.loc[:, "FVC"] = test["FVC"].values
submission.loc[:, "Confidence"] = test["Confidence"].values
submission = submission[["Patient_Week", "FVC", "Confidence"]]
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

                   Patient_Week          FVC  Confidence
0  ID00014637202177757139317_-3  3819.763184  269.480652
1  ID00014637202177757139317_-2  3818.903320  269.538971
2  ID00014637202177757139317_-1  3818.043213  269.597168
3   ID00014637202177757139317_0  3807.000000   70.000000
4   ID00014637202177757139317_1  3816.323486  269.713623
Wrote submission.csv with shape: (1908, 3)


/tmp/ipykernel_55/1388794381.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[3819.7632 3818.9033 3818.0432 ... 3122.0652 3121.2036 3120.3418]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.loc[:, "FVC"] = test["FVC"].values
/tmp/ipykernel_55/1388794381.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[269.48065 269.53897 269.59717 ... 226.99713 227.05627 227.11543]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.loc[:, "Confidence"] = test["Confidence"].values
